<a href="https://colab.research.google.com/github/mhsungur/Bitirme-Frontend/blob/main/spam_kontrol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer # Import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import FunctionTransformer # Import FunctionTransformer


from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,roc_auc_score,roc_curve

In [3]:
data=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Makine Öğrenimi/Kendi Kodlarım/spam_ham_dataset.csv',index_col='Unnamed: 0')

In [4]:
data.head()

,label,text,label_num
605,ham,Subject: enron methanol ; meter # : 988291\r\n...,0
2349,ham,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
3624,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0
4685,spam,"Subject: photoshop , windows , office . cheap ...",1
2030,ham,Subject: re : indian springs\r\nthis deal is t...,0


In [5]:
data.columns.to_list()

['label', 'text', 'label_num']

In [6]:
target='label'
x=data.drop(columns=[target,'label_num'])
y=data[target]

In [7]:
print(y.value_counts())

label
ham     3672
spam    1499
Name: count, dtype: int64


In [8]:
sozel_sutun=x.select_dtypes(include=["object"]).columns.to_list()
numeric_sutun=x.select_dtypes(exclude=["object"]).columns.to_list()

In [9]:
print(f"sözel sütunlar{sozel_sutun}")
print(f"sayısal sütun{numeric_sutun}")

sözel sütunlar['text']
sayısal sütun[]


In [10]:
x_train,x_test,y_train,y_test=train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
from sklearn.preprocessing import FunctionTransformer # Ensuring explicit import here
from sklearn.feature_extraction.text import TfidfVectorizer # Also ensuring this is imported if not already
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

sozel_transformer=Pipeline([
    ('selector', FunctionTransformer(lambda x: x.iloc[:, 0], validate=False)),
    ('tfidf', TfidfVectorizer())
])
numeric_transformer=StandardScaler()

preprocessor=ColumnTransformer(
    transformers=[("sozel",sozel_transformer, ['text']),
                  ("sayisal",numeric_transformer,numeric_sutun)]
)

In [12]:
classifier=LogisticRegression(max_iter=1000, class_weight='balanced') # Added class_weight='balanced'

model=Pipeline(steps=[
    ("preprocess",preprocessor),
    ("model",classifier)
])

In [13]:
model.fit(x_train,y_train)
y_pred=model.predict(x_test)
y_proba=model.predict_proba(x_test) # Removed [:1] to get probabilities for all samples

In [14]:
accuracy=accuracy_score(y_test,y_pred)
precision=precision_score(y_test,y_pred,pos_label="spam")
recall=recall_score(y_test,y_pred,pos_label="spam")
f1=f1_score(y_test,y_pred,pos_label="spam")


In [15]:

print("\n=== TEMEL METRİKLER ===")
print(f"Accuracy (Doğruluk) : {accuracy:.4f}")
print(f"Precision (Kesinlik): {precision:.4f}")
print(f"Recall (Duyarlılık) : {recall:.4f}")
print(f"F1-score           : {f1:.4f}")



=== TEMEL METRİKLER ===
Accuracy (Doğruluk) : 0.9652
Precision (Kesinlik): 0.8929
Recall (Duyarlılık) : 1.0000
F1-score           : 0.9434


In [16]:
cm = confusion_matrix(y_test, y_pred, labels=["ham", "spam"])
print("\n=== Karışıklık Matrisi (Confusion Matrix) ===")
print("Satırlar: Gerçek, Sütunlar: Tahmin (ham, spam)")
print(cm)


=== Karışıklık Matrisi (Confusion Matrix) ===
Satırlar: Gerçek, Sütunlar: Tahmin (ham, spam)
[[699  36]
 [  0 300]]
